# Task 1: Generative Storytelling App -
## CSC8646 - Generative AI Coursework

This notebook provides a side-by-side comparison of **Gemini 3.1 Pro** and **Gemini 3 Flash** for the MagicTales storytelling prototype. It demonstrates the iterative prompting process, narrative quality, and multimodal asset generation (Images and Audio).

## Development Environment and Supporting Documentation

The Generative Storytelling App was fully developed and implemented using Google AI Studio, where the complete interactive prototype was designed, tested, and refined. The application logic, structured prompts, temperature settings, and iterative prompt engineering were executed directly within Google AI Studio using the Gemini model.

The submitted .ipynb file is provided as a supporting technical document to demonstrate the prompt design, parameter configuration, experimental outputs, and validation process. It serves as documented evidence of the iterative development process rather than the primary application interface.

The actual working prototype—including user interaction flow, structured story generation, and output validation—is demonstrated in the recorded video submission. The notebook complements the prototype by transparently showing how prompts were engineered, tested, and improved to meet the assignment requirements.

In [4]:
# 1. Setup and Installation
!pip install -U -q google-genai

import os
import base64
import time
import IPython.display as display
from google import genai
from google.genai import types

# API Key Setup
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except ImportError:
    API_KEY = os.environ.get("GEMINI_API_KEY")

if not API_KEY:
    API_KEY = input("Please enter your Gemini API Key: ")

client = genai.Client(api_key=API_KEY)

## 2. Narrative Generation Comparison
We compare the high-reasoning **Gemini 3.1 Pro** (using System Instructions) against the high-speed **Gemini 3 Flash** (using a monolithic prompt).

In [2]:
system_instruction = """
You are a world-class children's storyteller. Create safe, engaging, and age-appropriate interactive stories.
Guidelines:
- Tone: Warm and non-scary.
- Safety: No violence or danger.
- Structure: Beginning, middle, end.
- Interactivity: End with exactly two choices (Choice A and Choice B).
- Length: 150-200 words.
"""

user_prompt = "Create a story for Leo (7) about a brave rabbit and a clever fox in a magical forest. Moral: Sharing makes friendships stronger."

print("--- Generating with Gemini 3.1 Pro (System Instructions) ---")
start_pro = time.time()
response_pro = client.models.generate_content(
    model="gemini-3.1-pro-preview",
    contents=user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        response_mime_type="application/json",
        response_schema={
            "type": "OBJECT",
            "properties": {
                "title": {"type": "STRING"},
                "story_text": {"type": "STRING"},
                "choices": {
                    "type": "OBJECT",
                    "properties": {
                        "Choice A": {"type": "STRING"},
                        "Choice B": {"type": "STRING"}
                    }
                }
            }
        }
    )
)
end_pro = time.time()
print(f"Latency: {end_pro - start_pro:.2f}s")
print(response_pro.text)

print("\n--- Generating with Gemini 3 Flash (Monolithic Prompt) ---")
start_flash = time.time()
response_flash = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=f"{system_instruction}\n\n{user_prompt}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        # Schema same as above
    )
)
end_flash = time.time()
print(f"Latency: {end_flash - start_flash:.2f}s")
print(response_flash.text)

--- Generating with Gemini 3.1 Pro (System Instructions) ---
Latency: 24.47s
{"title":"Barnaby and the Giant Sparkle-Berry","story_text":"In the magical Whispering Woods, a brave little rabbit named Barnaby discovered something amazing: a giant, glowing Sparkle-Berry! It was the biggest berry anyone had ever seen, shimmering with rainbow colors. Barnaby wanted to take it home, but it was far too heavy to carry alone. Just then, Felix the clever fox trotted by. Felix was not scary at all; he wore tiny reading glasses and loved building things. Seeing Barnaby struggle, Felix quickly gathered some twigs and leaves, building a sturdy little wagon to roll the heavy berry. Together, they pulled the wagon to a cozy clearing. Barnaby looked at the giant berry, his tummy rumbling. He could keep it all to himself, but he realized Felix had worked so hard to help him. Barnaby smiled and split the Sparkle-Berry right down the middle, giving a big piece to Felix. As they munched on the sweet, glowi

## 3. Multimodal Asset Generation
We now generate the visual and auditory assets for the first scene of the **Gemini 3.1 Pro** story.

In [3]:
story_data = response_pro.parsed

# 1. Scene Extraction
print("--- Extracting Scene 1 Visuals ---")
scene_response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=f"Identify the first visual scene for this story and provide a detailed image prompt. Story: {story_data['story_text']}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema={
            "type": "OBJECT",
            "properties": {
                "summary": {"type": "STRING"},
                "image_prompt": {"type": "STRING"}
            }
        }
    )
)
scene_data = scene_response.parsed
print(f"Scene Summary: {scene_data['summary']}")

# 2. Image Generation (Gemini 2.5 Flash Image)
print("\n--- Generating Illustration ---")
image_response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=f"storybook illustration style, colorful, child-friendly: {scene_data['image_prompt']}",
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(aspect_ratio="1:1")
    )
)

for part in image_response.candidates[0].content.parts:
    if part.inline_data:
        img_bytes = base64.b64decode(part.inline_data.data)
        display.display(display.Image(data=img_bytes))

# 3. Audio Generation (Gemini 2.5 Flash TTS)
print("\n--- Generating Narration ---")
audio_response = client.models.generate_content(
    model="gemini-2.5-flash-preview-tts",
    contents=[{"parts": [{"text": scene_data['summary']}]}],
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")
            )
        )
    )
)

audio_data = audio_response.candidates[0].content.parts[0].inline_data.data
audio_bytes = base64.b64decode(audio_data)
display.display(display.Audio(data=audio_bytes, rate=24000))

--- Extracting Scene 1 Visuals ---
Scene Summary: In the magical Whispering Woods, a brave little rabbit named Barnaby discovers a giant, glowing Sparkle-Berry that shimmers with vibrant rainbow colors.

--- Generating Illustration ---



--- Generating Narration ---


All the Actual Illustrations Narrations work in Google AI Studio


## 4. Comparative Analysis Summary
Based on the results above, we can conclude:
1. **Gemini 3.1 Pro** offers superior narrative depth and strictly adheres to the System Instructions, making it safer for children.
2. **Gemini 3 Flash** is significantly faster (lower latency) but may occasionally produce more generic or less structured content.
3. **Multimodal Integration** (Images and Audio) works seamlessly with the **Gemini 2.5 Flash** series, providing a rich, immersive experience for the child.